In [ ]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain.agents import AgentType, Tool, initialize_agent
from langchain.memory import ConversationBufferMemory
import os
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
from getpass import getpass

print("✅ All imports successful!")

/Users/dahliaelbanhawy/Desktop/IronHack/week_7/labs/lab-agent-vector-store/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
USER_AGENT environment variable not set, consider setting it to identify your requests.


✅ All imports successful!


In [ ]:

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ") #add openai key
os.environ["CHROMA_TELEMETRY"] = "False" #disable noisy warning

print("✅ Chroma telemetry disabled")


✅ Chroma telemetry disabled


In [15]:
# Build State of Union Vector Store
loader = TextLoader("state_of_the_union.txt")
documents = loader.load()

print(f"📄 Loaded {len(documents)} document(s)")

# Split into chunks
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)
texts = text_splitter.split_documents(documents)

print(f"📊 Split into {len(texts)} chunks")

# Create embeddings
embeddings = OpenAIEmbeddings(api_key=os.getenv('OPENAI_API_KEY'))

# Create vector store
docsearch = Chroma.from_documents(
    texts, 
    embeddings, 
    collection_name="state-of-union"
)

print("✅ State of Union vector store created successfully!")
print(f"📊 Vector store contains {docsearch._collection.count()} documents")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


📄 Loaded 1 document(s)
📊 Split into 42 chunks
✅ State of Union vector store created successfully!
📊 Vector store contains 84 documents


In [16]:
# Create RetrievalQA Chain

llm = OpenAI(temperature=0, api_key=os.getenv('OPENAI_API_KEY'))

state_of_union = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=docsearch.as_retriever()
)

print("✅ RetrievalQA chain created!")

✅ RetrievalQA chain created!


In [17]:
# Test the Vector Store
question = "What did Biden say about Ketanji Brown Jackson?"

result = state_of_union.run(question)
print(f"❓ Question: {question}")
print(f"✅ Answer: {result}")

❓ Question: What did Biden say about Ketanji Brown Jackson?
✅ Answer:  Biden nominated Ketanji Brown Jackson for the United States Supreme Court.


In [ ]:
# Build Ruff Vector Store
# Load the Ruff FAQ from the web
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")
docs = loader.load()

print(f"📄 Loaded {len(docs)} document(s)")

# Split into chunks using the same text splitter
ruff_texts = text_splitter.split_documents(docs)

print(f"📊 Split into {len(ruff_texts)} chunks")

# Create the Ruff vector store
ruff_db = Chroma.from_documents(
    ruff_texts, 
    embeddings,  # Reuse the same embeddings
    collection_name="ruff"
)

print("✅ Ruff vector store created successfully!")
print(f"📊 Vector store contains {ruff_db._collection.count()} documents")

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


📄 Loaded 1 document(s)
📊 Split into 23 chunks
✅ Ruff vector store created successfully!
📊 Vector store contains 69 documents


In [21]:
# Create Ruff RetrievalQA Chain
ruff = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=ruff_db.as_retriever()
)

print("✅ Ruff RetrievalQA chain created!")

✅ Ruff RetrievalQA chain created!


In [ ]:
# Test Ruff Vector Store
question = "Why use ruff over flake8?"

result = ruff.run(question)
print(f"❓ Question: {question}")
print(f"✅ Answer: {result}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


❓ Question: Why use ruff over flake8?
✅ Answer:  Ruff offers a larger rule set and the ability to automatically fix lint violations, while also being able to replace other tools such as Black, isort, and pyupgrade. It also supports a wider range of Python versions and does not require the installation of Rust.


In [23]:
# Compare Both Vector Stores
print("="*60)
print("STATE OF UNION QA SYSTEM")
print("="*60)
question1 = "What did Biden say about Ketanji Brown Jackson?"
result1 = state_of_union.run(question1)
print(f"❓ Q: {question1}")
print(f"✅ A: {result1}")

print("\n" + "="*60)
print("RUFF QA SYSTEM")
print("="*60)
question2 = "What are the benefits of using ruff?"
result2 = ruff.run(question2)
print(f"❓ Q: {question2}")
print(f"✅ A: {result2}")

STATE OF UNION QA SYSTEM
❓ Q: What did Biden say about Ketanji Brown Jackson?
✅ A:  Biden nominated Ketanji Brown Jackson for the United States Supreme Court.

RUFF QA SYSTEM
❓ Q: What are the benefits of using ruff?
✅ A: 
1. Consistent Code Formatting: Ruff's formatter ensures that your code is consistently formatted, making it easier to read and maintain.

2. Comprehensive Linting: Ruff's linter checks for a wide range of potential errors and style violations, helping to catch bugs and improve code quality.

3. Integration with Black: Ruff is compatible with Black, a popular code formatter, making it easy to use both tools together for even more consistent and error-free code.

4. Drop-in Replacement for Flake8: Ruff can be used as a drop-in replacement for Flake8, a popular code analysis tool, making it easy to switch to Ruff without losing any functionality.

5. Customizable Rules: Ruff allows you to customize the linting rules to fit your specific needs and preferences.

6. Python

In [24]:
# Create Tools

tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

print("✅ Tools created!")
print(f"📦 Total tools: {len(tools)}")
for tool in tools:
    print(f"  - {tool.name}")

✅ Tools created!
📦 Total tools: 2
  - State of Union QA System
  - Ruff QA System


In [25]:
# Create the Agent
agent = initialize_agent(
    tools, 
    llm, 
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    verbose=True,
    handle_parsing_errors=True
)

print("✅ Agent created successfully!")

✅ Agent created successfully!


/var/folders/88/scxzcgb55s1_sms0q1v2bkl80000gn/T/ipykernel_92245/606260814.py:2: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [ ]:
# Test State of Union Question
question = "What did Biden say about Ketanji Brown Jackson in the state of the union address?"

print(f"❓ Question: {question}")
print("="*60)
result = agent.invoke({"input": question})
print(f"✅ Answer: {result['output']}")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


❓ Question: What did Biden say about Ketanji Brown Jackson in the state of the union address?
 I should use the State of Union QA System to answer this question.
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the state of the union address?"
Observation:  Biden mentioned that he nominated Ketanji Brown Jackson to serve on the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.
Thought: I should use the Ruff QA System to check for any errors in the answer.
Action: Ruff QA System
Action Input: "Biden mentioned that he nominated Ketanji Brown Jackson to serve on the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."
Observation:  I don't know.
Thought: I should try rephrasing the question to see if I can get a better answer.
Action: State of Union QA System
Action

In [ ]:
# Test Ruff Question
question = "Why use ruff over flake8?"

print(f"❓ Question: {question}")
print("="*60)
result = agent.invoke({"input": question})
print(f"✅ Answer: {result['output']}")

# observation: The agent ended up answering a different question (not very far from the original) after the rephrasing attempt

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


❓ Question: Why use ruff over flake8?
 Ruff is a python linter that has some unique features compared to flake8, so it's worth exploring the differences.
Action: Ruff QA System
Action Input: "What are the unique features of ruff compared to flake8?"
Observation:  Ruff has a larger rule set, supports automatic fixing of lint violations, and does not support custom or third-party rules. It also has a formatter that is designed to be a drop-in replacement for Black.
Thought: These unique features could be beneficial for certain projects, but it's important to consider the trade-offs.
Action: State of Union QA System
Action Input: "What are the trade-offs of using ruff over flake8?"
Observation:  I don't know.
Thought: It seems like the State of Union QA System is not able to answer this question, so let's try asking Ruff QA System again.
Action: Ruff QA System
Action Input: "What are the trade-offs of using ruff over flake8?"
Observation: 
The main trade-off of using Ruff over Flake8 is t

In [ ]:
# Test Multi-Hop Question
question = "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"

print(f"❓ Question: {question}")
print("="*60)
result = agent.invoke({"input": question})
print(f"✅ Answer: {result['output']}")
#observation: perfect answer.

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


❓ Question: What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?
 I should check the documentation for ruff and the state of the union address.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA to run over Jupyter Notebooks.
Thought: I should now check the state of the union address.
Action: State of Union QA System
Action Input: Did the president mention nbQA in the state of the union?
Observation:  No, the president did not mention nbQA in the state of the union.
Thought: I now know the final answer.
Final Answer: The final answer is that Ruff uses nbQA to run over Jupyter Notebooks, but the president did not mention it in the state of the union.

> Finished chain.
✅ Answer: The final answer is that Ruff uses nbQA to run over Jupyter Notebooks, but the president did not mention it in the state of the union.


In [ ]:
# Router Tools with direct return
router_tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]


✅ Router tools created with return_direct=True


In [32]:
# Router Agent
router_agent = initialize_agent(
    router_tools, 
    llm, 
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    verbose=True,
    handle_parsing_errors=True
)

print("✅ Router agent created!")

✅ Router agent created!


In [33]:
# Test Router Agent with State of Union
question = "What did Biden say about Ketanji Brown Jackson?"

print(f"❓ Question: {question}")
print("="*60)
result = router_agent.invoke({"input": question})
print(f"✅ Answer: {result['output']}")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


❓ Question: What did Biden say about Ketanji Brown Jackson?
 It's important to know the context of the question and what information is being sought.
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson?"
Observation:  Biden nominated Ketanji Brown Jackson for the United States Supreme Court.


> Finished chain.
✅ Answer:  Biden nominated Ketanji Brown Jackson for the United States Supreme Court.


In [ ]:
# Compare Agent Behavior
print("="*70)
print("COMPARISON: NORMAL AGENT vs ROUTER AGENT")
print("="*70)

question = "Why use ruff over flake8?"

print("\n" + "─"*70)
print("1️⃣ NORMAL AGENT (with extra reasoning)")
print("─"*70)
normal_result = agent.invoke({"input": question})
print(f"Answer: {normal_result['output']}")

print("\n" + "─"*70)
print("2️⃣ ROUTER AGENT (returns directly)")
print("─"*70)
router_result = router_agent.invoke({"input": question})
print(f"Answer: {router_result['output']}")

#observation: Same end result with one question. 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


COMPARISON: NORMAL AGENT vs ROUTER AGENT

──────────────────────────────────────────────────────────────────────
1️⃣ NORMAL AGENT (with extra reasoning)
──────────────────────────────────────────────────────────────────────
 Ruff is a python linter that has some unique features compared to flake8, so it may be useful in certain situations.
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation:  Ruff offers a larger rule set and the ability to automatically fix lint violations, while also being able to replace other tools such as Black, isort, and pyupgrade. It also supports Python 3.13 and does not require the installation of Rust.
Thought: This information is helpful, but I should also consider the benefits of using flake8.
Action: Ruff QA System
Action Input: "What are the benefits of using flake8?"
Observation:  Flake8 is a popular Python linter that helps identify and fix coding errors and style violations in Python code. It can help improve code quality, rea

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I now have a better understanding of the differences between ruff and flake8.
Final Answer: Depending on the specific needs and preferences of a project, either ruff or flake8 may be a better choice for linting Python code. It is important to consider the features and benefits of each tool before making a decision.

> Finished chain.
Answer: Depending on the specific needs and preferences of a project, either ruff or flake8 may be a better choice for linting Python code. It is important to consider the features and benefits of each tool before making a decision.

──────────────────────────────────────────────────────────────────────
2️⃣ ROUTER AGENT (returns directly)
──────────────────────────────────────────────────────────────────────
 Ruff is a python linter that has some unique features compared to flake8, so it may be useful in certain situations.
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation:  Ruff offers a larger rule set and the ability to autom

In [ ]:
# Test Router Agent with Multi-Hop
question = "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"

print(f"❓ Question: {question}")
print("="*60)
result = router_agent.invoke({"input": question})
print(f"✅ Answer: {result['output']}")
# observation: agent completely disregarded the second part of the question because of the use of return-direct

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


❓ Question: What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?
 I should check the documentation for ruff and the state of the union address.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA to run over Jupyter Notebooks.


> Finished chain.
✅ Answer:  Ruff uses nbQA to run over Jupyter Notebooks.


In [ ]:
# Try a different agent type with the same router_tools

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)
conversational_agent = initialize_agent(
    router_tools,
    llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    memory=memory
)

# Test
question = "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
result = conversational_agent.invoke({"input": question})
print(f"✅ Answer: {result['output']}")

# Observation: Changing the agent did not help bypass the return_direct 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: Do I need to use a tool? Yes
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA to run over Jupyter Notebooks.


> Finished chain.
✅ Answer:  Ruff uses nbQA to run over Jupyter Notebooks.


In [ ]:
# Bonus: Router Agent with Memory

router_agent_with_memory = initialize_agent(
    router_tools,
    llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

# Test with conversation
print("="*60)
print("ROUTER AGENT WITH MEMORY")
print("="*60)

q1 = "What is ruff?"
print(f"\n❓ Q1: {q1}")
r1 = router_agent_with_memory.invoke({"input": q1})
print(f"✅ A1: {r1['output']}")

# Follow-up question (uses memory)
q2 = "Why is it better than flake8?"
print(f"\n❓ Q2: {q2}")
r2 = router_agent_with_memory.invoke({"input": q2})
print(f"✅ A2: {r2['output']}")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


ROUTER AGENT WITH MEMORY

❓ Q1: What is ruff?

Thought: Do I need to use a tool? Yes
Action: Ruff QA System
Action Input: What is ruff?

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Observation:  Ruff is a Python linter and formatter that can replace other tools such as Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade. It supports Python versions 3.7 and above and can be installed without needing to install Rust.


> Finished chain.
✅ A1:  Ruff is a Python linter and formatter that can replace other tools such as Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade. It supports Python versions 3.7 and above and can be installed without needing to install Rust.

❓ Q2: Why is it better than flake8?

Thought: Do I need to use a tool? Yes
Action: Ruff QA System
Action Input: Why is it better than flake8?
Observation:  Ruff is better than flake8 because it implements over 900 rules, compared to flake8's ~80 rules. Additionally, Ruff is capable of automatically fixing its own lint violations, while flake8 does not have this capability. Ruff also has better compatibility with other tools, such as type checkers, and